# Clasificación de residuos por material: CNN contra Vision Transformer
**Materia:** Inteligencia Artificial (DCI019), Doctorado en Ciencias e Ingeniería, UAG. **Autora:** Gabriela Medina Navidad.

Este cuaderno ejecuta el proyecto completo desde su repositorio, con la misma organización que el repositorio de la clase: el código vive en archivos `.py`, los parámetros en archivos YAML de la carpeta `configs/`, y aquí solo se corren comandos.

**Antes de empezar:** Entorno de ejecución, Cambiar tipo de entorno, GPU T4.

Los resultados se guardan en Google Drive, en `MyDrive/proyecto_IA/repo_resultados` y `MyDrive/proyecto_IA/repo_runs`, para que no se pierdan al cerrar la sesión.

### Celda 1. Importaciones y configuración

In [ ]:
import os
import glob
from IPython.display import Image, display

USUARIO_GITHUB = 'gnavidad1978'
RUTA_DRIVE = '/content/drive/MyDrive/proyecto_IA'

# Modelos a comparar: nombre de la red y su archivo de configuracion
MODELOS = {
    'mobilenet_v3_large': 'configs/mobilenet_v3.yaml',
    'yolo11n_cls': 'configs/yolo11n_cls.yaml',
    'resnet50': 'configs/resnet50.yaml',
    'vit_b_16': 'configs/vit_b_16.yaml',
}
SEMILLAS = [42, 7, 2026]   # para una prueba rapida basta [42]

### Celda 2. Conectar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Celda 3. Bajar el repositorio
Igual que en la clase, con `git clone`. Si ya estaba, se actualiza con la última versión. La última línea muestra la versión exacta del código que se está usando.

In [ ]:
%cd /content
if not os.path.exists('clasificacion_residuos_ia'):
    !git clone https://github.com/{USUARIO_GITHUB}/clasificacion_residuos_ia.git
else:
    !git -C clasificacion_residuos_ia pull
%cd /content/clasificacion_residuos_ia
!git log --oneline -1

### Celda 4. Instalar librerías
Las versiones están fijas en `requirements.txt` para que el cuaderno sea reproducible. PyTorch y torchvision son los de Colab.

In [ ]:
!pip install -q -r requirements.txt
import torch, torchvision, lightning
print('PyTorch', torch.__version__, '| torchvision', torchvision.__version__, '| Lightning', lightning.__version__,
      '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'sin GPU')

### Celda 5. Guardar resultados en Drive
La carpeta `resultados` del repositorio se enlaza con Drive para que los resultados de cada corrida queden guardados aunque se cierre la sesión. Los registros de TensorBoard (`runs`) se escriben en el disco de Colab, que se lee de inmediato, y se copian a Drive después de cada corrida.

In [ ]:
os.makedirs(f'{RUTA_DRIVE}/repo_resultados', exist_ok=True)
os.makedirs(f'{RUTA_DRIVE}/repo_runs', exist_ok=True)
if not os.path.islink('resultados'):
    !rm -rf resultados
    !ln -s "{RUTA_DRIVE}/repo_resultados" resultados
if not os.path.exists('runs'):
    !cp -r "{RUTA_DRIVE}/repo_runs" runs
print('resultados ->', os.path.realpath('resultados'))

### Celda 6. Preparar los datos
Usa el archivo de RealWaste que ya está en tu Drive (si no está, lo descarga), acomoda las clases a las seis categorías, agrupa las fotos casi idénticas, hace la partición 70, 15 y 15 con semilla fija, explora solo el entrenamiento y calcula la línea base. Es determinista: en cada sesión produce la misma partición. Tarda alrededor de un minuto.

In [ ]:
!python prepare_data.py --zip "{RUTA_DRIVE}/realwaste.zip"
display(Image('resultados/eda/conteo_entrenamiento.png'))
display(Image('resultados/eda/muestras_entrenamiento.png', width=600))

### Celda 7. TensorBoard
Ábrelo antes de entrenar. Si mientras entrena aparece *Data could not be loaded*, no afecta el entrenamiento; vuelve a correr esta celda al terminar la celda 8.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs

### Celda 8. Entrenar
Recorre todos los modelos y semillas de la celda 1 con un comando por corrida, por ejemplo `python main.py fit --config configs/vit_b_16.yaml --seed_everything 7`. Si una combinación ya tiene resultado en Drive, se salta; así, si Colab se desconecta, al volver a correr el cuaderno continúa donde se quedó.

In [ ]:
for backbone, config in MODELOS.items():
    for semilla in SEMILLAS:
        if os.path.exists(f'resultados/{backbone}_semilla{semilla}.json'):
            print(f'{backbone}, semilla {semilla}: ya entrenado, se salta')
            continue
        print(f'\n========== {backbone}, semilla {semilla} ==========')
        comando = f'python main.py fit --config {config} --seed_everything {semilla}'
        print(comando)
        !{comando}
        respaldo = f'cp -r runs/. "{RUTA_DRIVE}/repo_runs/"'
        !{respaldo}

### Celda 9. Comparación de modelos (validación)
Tabla con media ± desviación estándar entre semillas, curvas de entrenamiento y validación, matrices de confusión y fotos mal clasificadas. Todo se guarda en `resultados/figuras`.

In [ ]:
!python report.py
for figura in sorted(glob.glob('resultados/figuras/*.png')):
    print(figura)
    display(Image(figura, width=900 if 'curvas' in figura or 'errores' in figura else 450))

### Celda 10. Evaluación final con el conjunto de prueba
**Solo se corre una vez, al final del proyecto**, cuando ya no se van a cambiar modelos ni parámetros. Usa el mejor modelo de cada corrida guardado por el checkpoint. Quita el signo `#` de la última línea para ejecutarla.

In [ ]:
import json
def evaluar_prueba(backbone, semilla):
    r = json.load(open(f'resultados/{backbone}_semilla{semilla}.json'))
    comando = f'python main.py test --config {MODELOS[backbone]} --ckpt_path "{r["checkpoint"]}"'
    print(comando)
    get_ipython().system(comando)

# for b in MODELOS: evaluar_prueba(b, 42)